# 3. PINN для газового пласта: сравнение с методом конечных объёмов

В ноутбуке 1 PINN продолжила решение ОДУ за пределы данных, в ноутбуке 2 мы научились обучать PINN на уравнении в частных производных, у которого есть точное решение. Здесь задача ближе к практике: **разработка замкнутой газовой залежи одной скважиной**. Точного решения у неё нет, поэтому сначала мы строим **эталон** классическим методом, которым работают гидродинамические симуляторы, — методом конечных объёмов (КО), — и проверяем сам эталон. Затем обучаем PINN и сравниваем её с эталоном.

**Цель:** выяснить, может ли PINN, которая видит **только уравнения, начальное и граничные условия** (ни одного давления из расчёта КО), воспроизвести давление в пласте и дебит скважины за год с инженерной точностью.

Чем задача сложнее, чем в ноутбуке 2:

| | Ноутбук 2 | Этот ноутбук |
|---|---|---|
| уравнение | линейное, 1D по пространству | нелинейное (газ сжимаем), 2D осесимметричное $(r,z)$ + время |
| эталон | формула | численный расчёт, который тоже надо проверить |
| масштабы по пространству | один ($\delta/L\approx0.2$) | от 0.1 м до 1000 м: четыре порядка |
| масштабы по времени | 8 периодов | от секунд до года: семь порядков |
| физика | только фильтрация | фильтрация + гравитация, которая даёт слабый сигнал |

Приёмы из ноутбука 2 (обезразмеривание, масштабы из физики, жёсткие условия, Adam → L-BFGS) здесь применяются все сразу. Главный урок прежний: **малый лосс не означает точного решения**, точность проверяем только независимо.

## 1. Физическая задача

**Сценарий.** Газовая залежь толщиной $h=30$ м ограничена со всех сторон: кровля и подошва непроницаемы, на расстоянии $R=1000$ м от скважины — непроницаемая граница (например, разлом или выклинивание). В центре — вертикальная скважина радиуса $r_w=0.1$ м, вскрывающая пласт на всю толщину. Сначала пласт в покое, давление у подошвы $p_0=20$ МПа. Затем скважину запускают: за несколько часов забойное давление снижают до 15 МПа и дальше держат постоянным. Газ течёт к скважине, давление в пласте постепенно падает, дебит снижается.

**Что хотим найти:** поле давления $p(r,z,t)$ и **дебит** скважины (массовый расход газа) в течение года. Именно по таким расчётам планируют добычу: как быстро падает дебит, когда возмущение дойдёт до границы залежи, сколько газа отберём за год.

**Модель.** Однофазный газ, закон Дарси, однородная недеформируемая порода, осевая симметрия. Газ подчиняется уравнению реального газа с **постоянным** коэффициентом сверхсжимаемости $Z$:
$$
\rho=\frac{pM}{ZR_uT}=c\,p,\qquad c=\frac{M}{ZR_uT}.
$$
Закон Дарси с гравитацией (ось $z$ направлена вверх, $z=0$ — подошва) и закон сохранения массы:
$$
\mathbf j=-\frac{k\rho}{\mu}\big(\nabla p+\rho g\,\mathbf e_z\big),\qquad
\frac{\partial(\phi\rho)}{\partial t}+\nabla\cdot\mathbf j=0 .
$$
Плотность пропорциональна давлению, поэтому поток $\propto p\nabla p$ и уравнение **нелинейное**, в отличие от уравнения пьезопроводности для жидкости в ноутбуке 2.

**Что не учитываем:** воду, конденсат, зависимость $Z$ и $\mu$ от давления, неоднородность, турбулентное (недарсиевское) течение у скважины, ствол скважины и лифт. Это учебный бенчмарк, а не модель реального месторождения: зато оба решателя, КО и PINN, решают **одну и ту же** задачу и их можно честно сравнить.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import LogNorm
from matplotlib.patches import Rectangle
from scipy.integrate import solve_ivp
from scipy.interpolate import RegularGridInterpolator
from scipy.sparse import lil_matrix
from torch import nn

# float64: в невязке вторые производные и поправки от гравитации порядка 1e-3
torch.set_default_dtype(torch.float64)
torch.set_num_threads(2)
np.random.seed(7)
torch.manual_seed(7)
print("PyTorch:", torch.__version__)

# Цветовая схема: тема seaborn «whitegrid», палитра husl и три собственных цвета
PINN_COLOR = "#5b5fb0"  # индиго: PINN
REFERENCE_COLOR = "#e3242b"  # красный: эталон (КО), скважина
SAMPLES_COLOR = "#a1c9f4"  # пастельно-голубой: пласт, точки
MLP_COLOR = "#36ada4"  # husl, бирюзовый: сетка КО
INK_COLOR = "#262626"
GRID_COLOR = "#cccccc"
REFERENCE_STYLE = {"color": REFERENCE_COLOR, "linestyle": "--", "linewidth": 2.0}
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans", "Liberation Sans", "sans-serif"],
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.grid": True,
        "axes.axisbelow": True,
        "axes.edgecolor": GRID_COLOR,
        "axes.linewidth": 1.25,
        "axes.titlesize": 12,
        "axes.titlecolor": INK_COLOR,
        "axes.labelsize": 11,
        "axes.labelcolor": INK_COLOR,
        "grid.color": GRID_COLOR,
        "grid.linewidth": 1.0,
        "text.color": INK_COLOR,
        "xtick.color": INK_COLOR,
        "ytick.color": INK_COLOR,
        "xtick.bottom": False,
        "ytick.left": False,
        "legend.frameon": True,
        "legend.framealpha": 0.8,
        "legend.edgecolor": GRID_COLOR,
        "legend.fontsize": 9,
        "lines.solid_capstyle": "round",
    }
)
# Моменты времени для профилей давления: 1, 30, 180 и 365 суток
SHOW_DAYS = [1, 30, 180, 365]
TIME_COLORS = plt.cm.Blues(np.linspace(0.4, 0.95, len(SHOW_DAYS)))

### 1.1 Параметры пласта и газа

Газ — метан при 350 К, проницаемость 10 мД (типичный терригенный коллектор):

In [ ]:
# Пласт и газ (СИ)
mD = 9.869233e-16  # 1 миллидарси, м²
phi = 0.20  # пористость
k = 10 * mD  # проницаемость
mu = 2e-5  # вязкость газа, Па·с
T_res = 350.0  # пластовая температура, К
Z = 0.90  # коэффициент сверхсжимаемости
M = 0.01604  # молярная масса метана, кг/моль
Ru = 8.314462618  # универсальная газовая постоянная, Дж/(моль·К)
g = 9.80665  # ускорение свободного падения, м/с²

# Геометрия и режим скважины
p0 = 20e6  # начальное давление у подошвы, Па
rw, R, h = 0.1, 1000.0, 30.0  # радиус скважины, радиус залежи, толщина пласта, м
delta = 0.25  # депрессия в долях p0: забойное давление 20 → 15 МПа
t0 = 3600.0  # время запуска скважины, с
day = 86400.0
tend = 365 * day  # горизонт расчёта: год

c = M / (Z * Ru * T_res)  # rho = c p
beta = c * g  # обратная гидростатическая длина, 1/м
D = k * p0 / (phi * mu)  # пьезопроводность газа при p0, м²/с
L = np.log(R / rw)  # для логарифмической координаты по радиусу
S = np.log1p(tend / t0)  # для логарифмической координаты по времени


def well(t):
    # Забойное давление у подошвы в долях p0: плавно от 1 до 1 - delta за время ~t0
    return 1 - delta * (-np.expm1(-np.asarray(t) / t0)) ** 2


def a(z):
    # Гидростатический множитель: давление покоящегося газа p0 a(z)
    return np.exp(-beta * np.asarray(z))


# Моменты, в которых сохраняем решение: логарифмически от 1 с до года
times = np.unique(np.r_[0.0, np.geomspace(1.0, tend, 180), np.array(SHOW_DAYS) * day])

print(f"плотность газа у подошвы rho = c p0 = {c * p0:.1f} кг/м³")
print(
    f"гидростатический перепад по толщине пласта: {p0 * (1 - a(h)) / 1e3:.1f} кПа "
    f"(депрессия {delta * p0 / 1e6:.0f} МПа)"
)
print(f"пьезопроводность D = k p0 / (phi mu) = {D:.4f} м²/с")
print(f"возмущение доходит до границы за R²/(4D) ≈ {R**2 / (4 * D) / day:.0f} сут")

**Что мы видим.** Газ лёгкий: перепад давления от подошвы до кровли всего 36 кПа, меньше 1% депрессии 5 МПа. Гравитация в этой задаче — **слабый сигнал**, и это хороший тест: даже малая ошибка решателя может его исказить. Второе число — масштаб времени: волна депрессии доходит до границы залежи примерно за два месяца. Значит, за год мы увидим оба режима течения: сначала **неустановившийся** (пласт «не знает», что он ограничен), затем **истощение** всей залежи.

### 1.2 Убираем гидростатику заменой переменных

В покое давление газа не постоянно: оно убывает с высотой по гидростатике. Поскольку $\rho=cp$, условие равновесия $p_z=-\rho g=-\beta p$ даёт экспоненту, $\beta=cg$:
$$
p_{\text{гидр}}(z)=p_0\,a(z),\qquad a(z)=e^{-\beta z}.
$$
Будем искать давление в виде $p=p_0\,a(z)\,q(r,z,t)$. Тогда $q=1$ — это покой, а $1-q$ — относительная депрессия. Подставим в закон Дарси:
$$
\nabla p+\rho g\,\mathbf e_z=p_0a\nabla q+p_0q\,a'(z)\,\mathbf e_z+\beta p_0aq\,\mathbf e_z=p_0a\nabla q,
$$
потому что $a'=-\beta a$: **гравитационное слагаемое сокращается**. Поток $\mathbf j=-\frac{kcp_0^2}{\mu}a^2q\nabla q$, и уравнение баланса массы при постоянных свойствах принимает вид
$$
q_t=D\,a(z)\left[\frac1r\partial_r(r\,q\,q_r)+\partial_z(q\,q_z)-2\beta\,q\,q_z\right],\qquad D=\frac{kp_0}{\phi\mu}.
$$
Гравитация осталась в двух местах: множитель $a(z)$ и слагаемое $-2\beta qq_z$. Оба порядка $\beta h\approx2\cdot10^{-3}$.

**Начальное и граничные условия** в новой переменной:
- покой: $q(r,z,0)=1$;
- скважина: $q(r_w,z,t)=w(t)=1-\delta\big(1-e^{-t/t_0}\big)^2$. Забойное давление у подошвы плавно падает с 20 до 15 МПа за несколько часов, а по высоте ствола остаётся гидростатическим;
- непроницаемые внешняя граница, кровля и подошва: $q_r(R,z,t)=0$, $q_z(r,0,t)=q_z(r,h,t)=0$. В исходных переменных последнее условие — $p_z+\beta p=0$, а **не** $p_z=0$: на непроницаемой границе нулю равен поток, а не градиент давления.

Плавный запуск $w(t)$ выбран не случайно: $w(0)=1$ и $w'(0)=0$, поэтому условие на скважине **согласовано** с начальным, и у решения нет скачка в момент $t=0$ (ср. ноутбук 2, где начальное условие тоже согласовано с граничными).

Вывод с заменой переменных легко испортить знаком. Проверим его независимо: возьмём произвольное гладкое поле $q$, вычислим автоматическим дифференцированием невязку **исходного** уравнения для $p$ и невязку **преобразованного** уравнения для $q$ и сравним.

In [ ]:
def derivative(v, X):
    return torch.autograd.grad(v, X, torch.ones_like(v), create_graph=True)[0]


# Три точки (r, z, t) в физических единицах и «сфабрикованное» гладкое поле q
X = torch.tensor(
    [[0.3, 3.0, 1e4], [10.0, 15.0, 1e6], [500.0, 27.0, 1e7]], requires_grad=True
)
r, z, t = X.split(1, 1)
q = 1 + 0.03 * torch.sin(torch.log(r / rw) / L) * torch.cos(np.pi * z / h) * (t / tend)
p = p0 * torch.exp(-beta * z) * q
pd = derivative(p, X)
qd = derivative(q, X)

# Исходное уравнение: phi p_t + div(j) / c = 0, поток Дарси с гравитацией
Fr = -k / mu * p * pd[:, 0:1]
Fz = -k / mu * p * (pd[:, 1:2] + beta * p)
original = (
    phi * pd[:, 2:3] + derivative(r * Fr, X)[:, 0:1] / r + derivative(Fz, X)[:, 1:2]
) / (phi * p0 * torch.exp(-beta * z))

# Преобразованное уравнение для q
qr, qz, qt = qd.split(1, 1)
qrr = derivative(qr, X)[:, 0:1]
qzz = derivative(qz, X)[:, 1:2]
transformed = qt - D * torch.exp(-beta * z) * (
    qr * qr + q * qrr + q * qr / r + qz * qz + q * qzz - 2 * beta * q * qz
)
assert torch.allclose(original, transformed, rtol=1e-8, atol=1e-12)
print(f"невязка исходного уравнения:       {original.detach().ravel().numpy()}")
print(f"невязка преобразованного уравнения: {transformed.detach().ravel().numpy()}")
print(
    f"максимальное расхождение: {(original - transformed).abs().max().item():.1e} 1/с"
)

**Что мы видим.** Невязки обоих уравнений на произвольном поле совпадают до машинной точности: замена переменных выполнена верно. Такая проверка на «сфабрикованном» решении (method of manufactured solutions) не зависит ни от одного из решателей, поэтому ошибка в выводе не может спрятаться в обоих одинаково.

### 1.3 Режим скважины и два масштаба времени

Слева — забойное давление: скважину запускают за несколько часов, дальше давление постоянно. Справа — **радиус исследования** $\sqrt{4Dt}$: на какое расстояние от скважины успела дойти волна депрессии к моменту $t$. Пока он меньше $R$, течение неустановившееся, а когда достигает границы, начинается истощение всей залежи.

In [ ]:
t_plot = np.geomspace(60.0, tend, 400)
fig, ax = plt.subplots(1, 2, figsize=(14, 3.8), constrained_layout=True)

ax[0].semilogx(t_plot / day, p0 * well(t_plot) / 1e6, color=REFERENCE_COLOR, lw=2.5)
ax[0].axvline(t0 / day, color=INK_COLOR, ls=":", lw=1.5, label=r"$t_0$ = 1 ч")
ax[0].set(
    xlabel="время, сут",
    ylabel="давление, МПа",
    title="Забойное давление у подошвы $p_0\\,w(t)$",
)
ax[0].legend()

t_reach = R**2 / (4 * D)
ax[1].loglog(
    t_plot / day,
    np.sqrt(4 * D * t_plot),
    color=PINN_COLOR,
    lw=2.5,
    label=r"радиус исследования $\sqrt{4Dt}$",
)
ax[1].axhline(R, color=INK_COLOR, lw=2, label=f"граница залежи, R = {R:g} м")
ax[1].axvline(
    t_reach / day,
    color=INK_COLOR,
    ls=":",
    lw=1.5,
    label=f"$R^2/4D$ ≈ {t_reach / day:.0f} сут",
)
ax[1].set(
    xlabel="время, сут",
    ylabel="r, м",
    title="Куда дошла волна депрессии",
    ylim=(1, 5e3),
)
ax[1].legend(loc="lower right")
plt.show()

## 2. Геометрия: всё происходит у скважины

Скважина радиусом 0.1 м в залежи радиусом 1000 м — это отношение $10^{-4}$. В реальных пропорциях пласт — тонкий диск ($h/R=0.03$), а скважина на рисунке не видна. При этом падение давления и почти весь приток сосредоточены в первых метрах у скважины: поток газа через цилиндр радиуса $r$ одинаков, а площадь цилиндра растёт как $r$, поэтому градиент давления падает как $1/r$. Газ же почти весь находится далеко: объём кольца растёт как $r^2$.

Поэтому радиальную координату везде берём **логарифмической**:
$$
x=\frac{\ln(r/r_w)}{\ln(R/r_w)}\in[0,1].
$$
Ниже: пласт в масштабе, увеличенный участок у скважины с гранями грубой сетки КО и та же сетка в координатах $x$, $y=z/h$, в которых работает PINN.

In [ ]:
rf_demo = np.geomspace(
    rw, R, 64 + 1
)  # радиальные грани грубой сетки КО (64 × 8), см. раздел 3
zf_demo = np.linspace(0, h, 8 + 1)

# Разрез (r, z) в реальных пропорциях
fig, ax = plt.subplots(figsize=(13, 1.9), constrained_layout=True)
ax.add_patch(Rectangle((rw, 0), R - rw, h, color=SAMPLES_COLOR, alpha=0.6, lw=0))
ax.plot(
    [rw, rw], [0, h], color=REFERENCE_COLOR, lw=4, label=f"скважина, $r_w$ = {rw} м"
)
ax.axvline(R, color=INK_COLOR, lw=2, label=f"непроницаемая граница, $R$ = {R:g} м")
ax.set(
    xlim=(-20, R + 20),
    ylim=(-8, h + 8),
    aspect="equal",
    xlabel="r, м",
    ylabel="z, м",
    title=f"Разрез пласта в масштабе: {R:g} × {h:g} м",
)
ax.legend(loc="center right", fontsize=8)
plt.show()

# Окрестность скважины и та же сетка в координатах PINN
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2), constrained_layout=True)
for r_ in rf_demo[rf_demo <= 5]:
    ax[0].axvline(r_, color=MLP_COLOR, lw=0.8)
for z_ in zf_demo:
    ax[0].axhline(z_, color=MLP_COLOR, lw=0.8)
ax[0].axvline(rw, color=REFERENCE_COLOR, lw=4, label="скважина")
ax[0].set(
    xlim=(0, 5),
    ylim=(0, h),
    xlabel="r, м",
    ylabel="z, м",
    title="Первые 5 м: грани ячеек КО (z не в масштабе)",
)
ax[0].legend(loc="lower right")
ax[0].grid(False)

for x_ in np.log(rf_demo / rw) / L:
    ax[1].axvline(x_, color=MLP_COLOR, lw=0.6)
for y_ in zf_demo / h:
    ax[1].axhline(y_, color=MLP_COLOR, lw=0.6)
ax[1].axvline(0, color=REFERENCE_COLOR, lw=4)
ax[1].axvline(1, color=INK_COLOR, lw=2)
for r_ in (1, 10, 100, 1000):
    ax[1].text(
        np.log(r_ / rw) / L,
        1.03,
        f"{r_} м",
        ha="right" if r_ == 1000 else "center",
        fontsize=9,
    )
ax[1].set(
    xlim=(-0.02, 1.02),
    ylim=(0, 1.1),
    xlabel=r"$x=\ln(r/r_w)/L$",
    ylabel=r"$y=z/h$",
    title="Координаты PINN: та же сетка равномерна по x",
)
ax[1].grid(False)
plt.show()

for r_ in (1, 10, 100):
    print(
        f"доля объёма пласта при r < {r_:>3} м: {(r_**2 - rw**2) / (R**2 - rw**2):.4%}"
    )

**Что мы видим.** Каждый десятичный порядок по радиусу занимает четверть отрезка $x\in[0,1]$, но содержит в 100 раз больше газа, чем предыдущий: 99% запасов лежит при $r>100$ м. Это противоречие ещё всплывёт: область, где решение меняется быстро (у скважины), и область, где находится газ (далеко), — разные.

> **Обсуждение.** Почему в задаче нет условия на оси $r=0$? Что изменилось бы, если бы мы моделировали скважину точечным источником?

## 3. Эталон: метод конечных объёмов

Так считают гидродинамические симуляторы. Пласт делим на ячейки-кольца: 64 по радиусу (равномерно по $\ln r$) и 8 по высоте. Для каждой ячейки записываем **баланс массы**: изменение массы газа в ячейке равно сумме потоков через её грани,
$$
\phi\,c\,p_0\,a_i\,V_i\,\frac{dq_i}{dt}=-\sum_{\text{грани }f}F_{if}.
$$
Поток через грань считаем по закону Дарси из значений $q$ в соседних ячейках. Каждый поток входит в баланс двух соседних ячеек с противоположными знаками, поэтому газ не появляется и не исчезает: схема **консервативна**. Ещё несколько деталей, которые делают эталон надёжным:
- объёмы ячеек — точные объёмы колец, радиальная проводимость грани — точная для цилиндра (логарифм отношения радиусов);
- множитель $a^2$ считается на гранях, поэтому покой $q=1$ — **точное** равновесие дискретной схемы;
- среднее арифметическое $\bar q$ на грани даёт $\bar q\,\Delta q=\Delta(q^2)/2$, то есть дискретный аналог $q\,q_r=(q^2)_r/2$.

Получается система ОДУ по времени: 512 уравнений на сетке 64×8 и 2048 на сетке 128×16. Она **жёсткая**: у скважины ячейки маленькие и реагируют за доли секунды, у границы — большие и медленные. Её интегрирует неявный метод BDF из SciPy с адаптивным шагом. Дополнительная переменная копит массу газа, ушедшую в скважину, чтобы проверить баланс.

In [ ]:
def make_fv(nr=64, nz=8):
    rf = np.geomspace(rw, R, nr + 1)  # радиальные грани
    rc = np.sqrt(rf[:-1] * rf[1:])  # центры ячеек (среднее геометрическое)
    zf = np.linspace(0, h, nz + 1)
    zc = (zf[:-1] + zf[1:]) / 2
    dz = h / nz
    ann = np.pi * np.diff(rf**2)  # площади колец
    V = np.broadcast_to(ann[:, None] * dz, (nr, nz)).copy()
    storage = phi * c * p0 * a(zc)[None, :] * V
    Cr = 2 * np.pi * dz * k * c * p0**2 / mu * a(zc) ** 2
    Cz = ann[:, None] * k * c * p0**2 / mu * a(zf[1:-1])[None, :] ** 2 / dz

    def fluxes(t, q):
        # Массовые потоки через грани; положительный fr направлен от скважины
        fr = np.zeros((nr + 1, nz))
        fz = np.zeros((nr, nz + 1))  # кровля и подошва: нулевой поток
        fr[0] = -Cr * 0.5 * (q[0] + well(t)) * (q[0] - well(t)) / np.log(rc[0] / rw)
        fr[1:nr] = (
            -Cr[None, :]
            * 0.5
            * (q[1:] + q[:-1])
            * np.diff(q, axis=0)
            / np.log(rc[1:] / rc[:-1])[:, None]
        )
        fz[:, 1:nz] = -Cz * 0.5 * (q[:, 1:] + q[:, :-1]) * np.diff(q, axis=1)
        return fr, fz  # fr[nr] = 0: внешняя граница непроницаема

    def rhs(t, state):
        q = state[:-1].reshape(nr, nz)
        fr, fz = fluxes(t, q)
        dq = -(np.diff(fr, axis=0) + np.diff(fz, axis=1)) / storage
        return np.r_[dq.ravel(), -fr[0].sum()]  # последняя переменная: добытая масса

    # Структура якобиана: ячейка зависит только от себя и четырёх соседей
    n = nr * nz
    sp = lil_matrix((n + 1, n + 1), dtype=int)
    for i in range(nr):
        for j in range(nz):
            row = i * nz + j
            for ii, jj in [(i, j), (i - 1, j), (i + 1, j), (i, j - 1), (i, j + 1)]:
                if 0 <= ii < nr and 0 <= jj < nz:
                    sp[row, ii * nz + jj] = 1
    sp[n, :nz] = 1
    return dict(
        nr=nr,
        nz=nz,
        rf=rf,
        rc=rc,
        zc=zc,
        V=V,
        storage=storage,
        fluxes=fluxes,
        rhs=rhs,
        sparsity=sp.tocsr(),
    )


def run_fv(nr, nz, rtol=2e-7):
    mesh = make_fv(nr, nz)
    n = nr * nz
    sol = solve_ivp(
        mesh["rhs"],
        (0, tend),
        np.r_[np.ones(n), 0.0],  # покой: q = 1, добытая масса 0
        method="BDF",
        t_eval=times,
        rtol=rtol,
        atol=np.r_[np.full(n, 1e-9), 1e-3],
        jac_sparsity=mesh["sparsity"],
        max_step=tend / 100,
    )
    assert sol.success, sol.message
    mesh["q"] = sol.y[:-1].T.reshape(-1, nr, nz)  # (время, r, z)
    mesh["cumulative"] = sol.y[-1]  # добытая масса, кг
    mesh["rate"] = np.array(
        [-mesh["fluxes"](t, q)[0][0].sum() for t, q in zip(times, mesh["q"])]
    )  # дебит, кг/с
    mesh["mass"] = (mesh["q"] * mesh["storage"]).sum(axis=(1, 2))  # газ в пласте, кг
    assert mesh["q"].min() > 0
    return mesh


start = time.time()
fv = run_fv(64, 8)  # грубая сетка: на ней сравниваем с PINN
fine = run_fv(128, 16)  # тонкая сетка: эталон
print(f"расчёт КО на двух сетках: {time.time() - start:.1f} с")

### 3.1 Проверяем сам эталон

Эталону нельзя верить на слово. Три независимые проверки:
1. **равновесие:** при $q=1$ и неработающей скважине правая часть системы должна быть нулём;
2. **баланс массы:** газ в пласте плюс добытый газ равны начальным запасам;
3. **измельчение сетки:** решения на сетках 64×8 и 128×16 должны почти совпадать. Это не доказательство сходимости, а оценка порядка ошибки эталона.

In [ ]:
check = make_fv(16, 4)
equilibrium = np.max(np.abs(check["rhs"](0, np.r_[np.ones(16 * 4), 0.0])))
assert equilibrium < 1e-15

# Решение тонкой сетки в центрах грубой сетки (интерполяция по t, ln r, z)
interp = RegularGridInterpolator(
    (times, np.log(fine["rc"]), fine["zc"]), fine["q"], bounds_error=True
)
TT, RR, ZZ = np.meshgrid(times, np.log(fv["rc"]), fv["zc"], indexing="ij")
qfine = interp(np.c_[TT.ravel(), RR.ravel(), ZZ.ravel()]).reshape(fv["q"].shape)
refine_max = float(np.max(np.abs(fv["q"] - qfine)) * p0 / 1e3)
rate_refine = float(
    np.linalg.norm(fv["rate"] - fine["rate"]) / np.linalg.norm(fine["rate"])
)

m0 = fine["mass"][0]
balance = np.max(np.abs(fine["mass"] + fine["cumulative"] - m0)) / m0
# Та же проверка, но добытая масса — интеграл сохранённых дебитов по трапециям
trap = np.r_[
    0, np.cumsum(0.5 * (fine["rate"][1:] + fine["rate"][:-1]) * np.diff(times))
]
trap_balance = np.max(np.abs(fine["mass"] + trap - m0)) / m0
assert balance < 1e-5

print(f"1. правая часть в покое:                    {equilibrium:.1e}")
print(f"2. невязка баланса массы / запасы:          {balance:.1e}")
print(f"   то же, дебиты по трапециям:              {trap_balance:.1e}")
print(
    f"3. разница сеток: давление, макс.           {refine_max:.2f} кПа "
    f"({refine_max / (delta * p0 / 1e3):.3%} депрессии)"
)
print(f"   разница сеток: дебит, отн. L2            {rate_refine:.3%}")

**Что мы видим.** Покой — точное равновесие, баланс массы выполняется до машинной точности (по трапециям — с ошибкой квадратуры по времени ~$10^{-6}$). Две сетки расходятся по давлению примерно на 1 кПа, то есть на 0.02% депрессии, а по дебиту на сотые доли процента. Значит, ошибку PINN порядка десятков кПа эталон измерить способен. На 512 ячейках расчёт занимает доли секунды.

### 3.2 Как выглядит решение

In [ ]:
rho_std = 0.668  # плотность метана при стандартных условиях (20 °C, 1 атм), кг/м³
j_mid = fine["nz"] // 2
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2), constrained_layout=True)

for days, col in zip(SHOW_DAYS, TIME_COLORS):
    it = np.argmin(abs(times - days * day))
    ax[0].semilogx(
        fine["rc"],
        p0 * a(fine["zc"][j_mid]) * fine["q"][it, :, j_mid] / 1e6,
        color=col,
        lw=2.5,
        label=f"{days} сут",
    )
ax[0].set(
    xlabel="r, м", ylabel="давление, МПа", title="Давление в середине пласта по высоте"
)
ax[0].legend(loc="lower right")

late = (
    times >= 60.0
)  # в первую минуту дебит ничтожен, и там виден шум допуска интегратора
ax[1].semilogx(times[late] / day, fine["rate"][late], color=REFERENCE_COLOR, lw=2.5)
ax[1].axvline(R**2 / (4 * D) / day, color=INK_COLOR, ls=":", lw=1.5, label="$R^2/4D$")
ax[1].set(xlabel="время, сут", ylabel="дебит, кг/с", title="Дебит скважины")
ax[1].legend()

ax[2].semilogx(
    times[late] / day, 100 * fine["cumulative"][late] / m0, color=PINN_COLOR, lw=2.5
)
ax[2].axhline(
    100 * delta, color=INK_COLOR, ls="--", lw=1.2, label="предел при $q\\to w(\\infty)$"
)
ax[2].set(xlabel="время, сут", ylabel="% начальных запасов", title="Отобрано газа")
ax[2].legend(loc="upper left")
plt.show()

for days in SHOW_DAYS:
    it = np.argmin(abs(times - days * day))
    print(
        f"{days:>3} сут: дебит {fine['rate'][it]:6.3f} кг/с "
        f"≈ {fine['rate'][it] * day / rho_std / 1e3:6.1f} тыс. ст. м³/сут, "
        f"отобрано {fine['cumulative'][it] / m0:6.2%} запасов"
    )

**Что мы видим.**
- **Запуск.** Пока забойное давление снижается (первые часы), дебит растёт от нуля до максимума около 8.3 кг/с.
- **Первые два месяца: неустановившийся режим.** Воронка депрессии расширяется от скважины: через сутки она дошла примерно до 150 м, а давление у границы ещё 20 МПа. Дебит медленно падает (7.2 кг/с через сутки, 5.8 кг/с через месяц): перепад давления тот же, а газу приходится идти всё дальше.
- **Дальше — истощение.** Волна дошла до непроницаемой границы, и давление падает уже во всей залежи: у границы 19.4 МПа через полгода и 18.7 МПа через год. Дебит падает заметно быстрее, теперь из-за снижения пластового давления, и к концу года он 4.1 кг/с, около 530 тыс. м³/сут при стандартных условиях.
- **Предел.** Если ждать бесконечно, пласт выровняется до забойного давления: $q\to w(\infty)=0.75$, дебит станет нулевым, а отобрано будет 25% запасов (газ сжимаем: при снижении давления на четверть из пласта уходит четверть массы). За год до предела ещё далеко.

Это и есть эталон, с которым будем сравнивать PINN.

## 4. PINN: только уравнение и условия

### 4.1 Безразмерные координаты из масштабов задачи

Как в ноутбуке 2, сначала приводим всё к величинам порядка единицы. Масштабы здесь логарифмические, поэтому и координаты логарифмические:
$$
x=\frac{\ln(r/r_w)}{\ln(R/r_w)},\qquad y=\frac zh,\qquad s=\frac{\ln(1+t/t_0)}{\ln(1+t_{\text{end}}/t_0)},\qquad x,y,s\in[0,1].
$$
По $s$ одинаковые отрезки покрывают и первые часы после запуска, и последние месяцы года. В новых координатах уравнение для $q$ (цепное правило, $r\,\partial_r=\partial_x/L$):
$$
\frac{q_s}{S\,(t+t_0)}=D\,a\left[\frac{q_x^2+q\,q_{xx}}{r^2L^2}+\frac{q_y^2+q\,q_{yy}-2\beta h\,q\,q_y}{h^2}\right],\qquad L=\ln\frac R{r_w},\quad S=\ln\Big(1+\frac{t_{\text{end}}}{t_0}\Big).
$$

### 4.2 Все условия встроены в сеть

Как на шаге 4 ноутбука 2, условия выполняются **точно** за счёт формы решения:
$$
q=1-\delta\big(1-e^{-t/t_0}\big)^2 f,\qquad f=\exp\big\{-\xi\,\operatorname{softplus}(4+N)\big\},\qquad \xi=2x-x^2 .
$$
- $t=0$: множитель $(1-e^{-t/t_0})^2=0$, поэтому $q=1$ — начальное условие;
- $x=0$ (скважина): $\xi=0$, $f=1$, поэтому $q=w(t)$;
- $x=1$ (внешняя граница): $\partial_x\xi=2-2x=0$, сеть зависит от $x$ только через $\xi$, поэтому $q_x=0$;
- $y=0$ и $y=1$ (подошва и кровля): от высоты сеть зависит только через $\cos\pi y$, производная которого равна нулю на обеих границах, поэтому $q_y=0$;
- $0<f\le1$, поэтому $0.75\le q\le1$: давление не может выйти за физически разумные пределы. При этом $f$ на внешней границе может быть заметно больше нуля — давление там падает.

Сеть $N$ состоит из двух частей: основная зависит от $(\xi,s)$ и описывает радиальное течение, а маленькая поправка от $(\xi,s,\cos\pi y)$ умножена на $\beta h\approx2\cdot10^{-3}$ — масштаб, в котором гравитация меняет решение по высоте. Последние слои обеих сетей инициализированы нулями, так что до обучения $f=e^{-4.02\,\xi}$: депрессия сосредоточена у скважины. Это разумное начальное приближение.

### 4.3 Нормировка невязки

Слагаемые уравнения различаются на порядки: у скважины в первые минуты $D/(r^2L^2)\sim10^{-1}$ 1/с, а на границе через год $1/[S(t+t_0)]\sim10^{-8}$ 1/с. Без нормировки лосс видел бы только окрестность скважины. Делим невязку на сумму масштабов временного и радиального слагаемых,
$$
\hat{\mathcal R}=\frac{\mathcal R}{1/[S(t+t_0)]+D/(r^2L^2)}.
$$
Нули невязки при этом не меняются, меняется только то, какие области оптимизатор считает важными. Лосс — среднее $\hat{\mathcal R}^2$ по точкам коллокации; **никаких данных из расчёта КО в нём нет.**

| Приём из ноутбука 2 | Как он применён здесь |
|---|---|
| обезразмеривание | $q=p/(p_0a)$, координаты $x,y,s\in[0,1]$ |
| масштабы из физики | логарифмические координаты по $r$ и $t$, поправка с масштабом $\beta h$ |
| жёсткие условия | начальное условие и **все** граничные условия встроены в $q$ |
| Adam → L-BFGS | 5000 шагов Adam, затем 300 итераций L-BFGS |
| новое | нормировка невязки на локальный масштаб |

### 4.4 Где лежат точки коллокации

Точки коллокации берём из последовательности Соболя (квазислучайная, покрывает куб равномернее случайных точек) равномерно по $(x,y,s)$, то есть равномерно по $\ln r$, $z$ и $\ln(1+t/t_0)$. В физических координатах они скапливаются у скважины и на ранних временах. Так и задумано: там решение меняется быстрее всего. Но газ, а значит и баланс массы, и поздний дебит, — далеко и поздно. Сравним долю точек с долей объёма пласта и долей времени.

Первые 4096 точек этой последовательности — независимый тестовый набор `Xtest`, оптимизатор их не видит. У теста та же неравномерность, что у обучения.

In [ ]:
eng = torch.quasirandom.SobolEngine(
    3, scramble=True, seed=12
)  # тот же seed, что при обучении
Xs = eng.draw(4096).numpy()
r_s = rw * np.exp(L * Xs[:, 0])
t_s = t0 * np.expm1(S * Xs[:, 2])

r_edges = np.array([rw, 1.0, 10.0, 100.0, R])
r_lab = ["0.1–1 м", "1–10 м", "10–100 м", "100–1000 м"]
pts_r = np.histogram(r_s, r_edges)[0] / len(r_s)
vol_r = np.diff(r_edges**2) / (R**2 - rw**2)

t_edges = np.array([0.0, 3600.0, day, 30 * day, 180 * day, tend])
t_lab = ["0–1 ч", "1 ч–1 сут", "1–30 сут", "30–180 сут", "180–365 сут"]
pts_t = np.histogram(t_s, t_edges)[0] / len(t_s)
dur_t = np.diff(t_edges) / tend

fig, ax = plt.subplots(1, 3, figsize=(17, 4.4), constrained_layout=True)
n_show = 1024
ax[0].loglog(
    r_s[:n_show],
    t_s[:n_show] / day,
    ".",
    ms=3,
    color=PINN_COLOR,
    alpha=0.6,
    label=f"{n_show} точек коллокации",
)
tt = np.geomspace(1.0, tend, 200)
ax[0].loglog(
    np.sqrt(4 * D * tt),
    tt / day,
    color=REFERENCE_COLOR,
    lw=2,
    label=r"радиус исследования $\sqrt{4Dt}$",
)
ax[0].axvline(R, color=INK_COLOR, lw=1.5, label="внешняя граница")
ax[0].set(
    xlim=(rw / 1.5, R * 1.5),
    ylim=(1e-4, 600),
    xlabel="r, м",
    ylabel="t, сут",
    title="Точки коллокации в физических (r, t)",
)
ax[0].legend(fontsize=8, loc="upper left", framealpha=0.95)

for a_, lab, pts, ref, ref_lab, title in [
    (ax[1], r_lab, pts_r, vol_r, "доля объёма пласта", "Радиус: точки и газ"),
    (ax[2], t_lab, pts_t, dur_t, "доля 365 суток", "Время: точки и длительность"),
]:
    xb = np.arange(len(lab))
    a_.bar(xb - 0.2, pts, 0.4, color=PINN_COLOR, label="доля точек коллокации")
    a_.bar(xb + 0.2, ref, 0.4, color=SAMPLES_COLOR, edgecolor=INK_COLOR, label=ref_lab)
    a_.set(
        xticks=xb,
        xticklabels=lab,
        yscale="log",
        ylim=(1e-5, 300),
        title=title,
        ylabel="доля",
    )
    a_.legend(fontsize=8, loc="upper left")
plt.show()

print("интервал r      точки   объём пласта")
for lab, p_, v_ in zip(r_lab, pts_r, vol_r):
    print(f"{lab:<13} {p_:7.1%}   {v_:10.4%}")
print("интервал t      точки   доля времени")
for lab, p_, v_ in zip(t_lab, pts_t, dur_t):
    print(f"{lab:<13} {p_:7.1%}   {v_:10.1%}")

**Что мы видим.** На внешнее кольцо 100–1000 м, где лежит 99% газа, приходится только четверть точек, а на последние полгода — меньше 8%. Если PINN где-то и ошибётся, то скорее всего далеко от скважины и поздно. Проверим это после обучения.

### 4.5 Обучение

Сеть небольшая: основная часть — 3 скрытых слоя по 40 нейронов, поправка — 3 слоя по 24. На каждом шаге Adam берём 1024 новые точки Соболя, затем L-BFGS дожимает на фиксированном наборе из 4096 точек. Обучение занимает около минуты.

In [ ]:
ADAM_STEPS = 5000
LBFGS_STEPS = 300
N_COLLOCATION = 1024


class MLP(nn.Module):
    def __init__(self, nin, width=40):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(nin, width),
            nn.Tanh(),
            nn.Linear(width, width),
            nn.Tanh(),
            nn.Linear(width, width),
            nn.Tanh(),
            nn.Linear(width, 1),
        )
        # Нулевой последний слой: до обучения N = 0
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, x):
        return self.net(x)


class PINN(nn.Module):
    # (x, y, s) -> q. Начальное условие, скважина и непроницаемые границы выполняются точно
    def __init__(self):
        super().__init__()
        self.base = MLP(2)  # радиальное течение: (xi, s)
        self.correction = MLP(3, 24)  # поправка по высоте: (xi, s, cos πy)

    def forward(self, X):
        x, y, s = X.split(1, dim=1)
        xi = 2 * x - x * x
        t = t0 * torch.expm1(S * s)
        N = self.base(torch.cat([xi, s], 1)) + beta * h * self.correction(
            torch.cat([xi, s, torch.cos(np.pi * y)], 1)
        )
        f = torch.exp(-xi * torch.nn.functional.softplus(4 + N))
        return 1 - delta * (-torch.expm1(-t / t0)) ** 2 * f


model = PINN()


def grad(v, X):
    return torch.autograd.grad(v, X, torch.ones_like(v), create_graph=True)[0]


def residual(X, net=model):
    # Нормированная невязка уравнения для q в координатах (x, y, s)
    X = X.detach().clone().requires_grad_(True)
    q = net(X)
    qx, qy, qs = grad(q, X).split(1, 1)
    qxx = grad(qx, X)[:, 0:1]
    qyy = grad(qy, X)[:, 1:2]
    x, y, s = X.split(1, 1)
    r = rw * torch.exp(L * x)
    t = t0 * torch.expm1(S * s)
    radial = (qx * qx + q * qxx) / (r * r * L * L)
    vertical = (qy * qy + q * qyy - 2 * beta * h * q * qy) / (h * h)
    scale = 1 / (S * (t + t0)) + D / (r * r * L * L)
    return (
        qs / (S * (t + t0)) - D * torch.exp(-beta * h * y) * (radial + vertical)
    ) / scale


sobol = torch.quasirandom.SobolEngine(3, scramble=True, seed=12)
Xtest = sobol.draw(4096).double()  # независимый тест: оптимизатор его не видит
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []
start = time.time()
for step in range(ADAM_STEPS):
    X = sobol.draw(N_COLLOCATION).double()  # новые точки на каждом шаге
    opt.zero_grad()
    loss = (residual(X) ** 2).mean()
    if not torch.isfinite(loss):
        raise RuntimeError(
            "Лосс PINN не конечен (inf/NaN); уменьшите скорость обучения."
        )
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
    opt.step()
    history.append(loss.item())
    if (step + 1) % 1000 == 0:
        print(f"Adam {step + 1}: лосс {history[-1]:.2e}")

# L-BFGS нужен фиксированный набор точек для линейного поиска
Xfixed = sobol.draw(4096).double()
opt2 = torch.optim.LBFGS(
    model.parameters(),
    lr=0.5,
    max_iter=LBFGS_STEPS,
    history_size=50,
    line_search_fn="strong_wolfe",
    tolerance_grad=1e-10,
)


def closure():
    opt2.zero_grad()
    loss = (residual(Xfixed) ** 2).mean()
    loss.backward()
    return loss


opt2.step(closure)
lbfgs_loss = closure().item()
test_rms = (residual(Xtest) ** 2).mean().sqrt().item()
print(f"после L-BFGS: лосс {lbfgs_loss:.2e}")
print(
    f"обучение {time.time() - start:.0f} с; нормированная невязка на тесте (RMS) {test_rms:.2e}"
)

## 5. Независимая проверка

Лосс мал, но, как мы видели в ноутбуке 2, это ничего не гарантирует. Сравниваем PINN с эталоном в центрах ячеек грубой сетки КО (эталон интерполирован с тонкой сетки) и смотрим на инженерные величины:
- **давление:** максимальная ошибка в кПа и в долях депрессии $\delta p_0=5$ МПа. Ошибка 1% от абсолютного давления 20 МПа — это 200 кПа, то есть уже 4% депрессии, поэтому нормируем на депрессию;
- **дебит:** считаем по закону Дарси из $q_x$ на стенке скважины, производная — автоматическим дифференцированием, интеграл по высоте — квадратурой Гаусса — Лежандра. Дебит зависит от **производной** решения и обычно сложнее давления;
- **баланс массы:** запасы газа в PINN (сумма по ячейкам тонкой сетки) плюс добытый газ (интеграл дебита PINN) должны быть постоянны. PINN не консервативна по построению, так что это настоящая проверка.

Цели, которые мы ставим заранее: ошибка давления меньше 5% депрессии, объёмно-взвешенная относительная ошибка давления меньше 1%, ошибка дебита меньше 10%, невязка баланса массы меньше 1% запасов.

In [ ]:
def predict_q(t, r, z):
    t, r, z = np.broadcast_arrays(t, r, z)
    X = np.c_[np.log(r.ravel() / rw) / L, z.ravel() / h, np.log1p(t.ravel() / t0) / S]
    with torch.no_grad():
        out = np.concatenate(
            [
                model(torch.tensor(b)).numpy().ravel()
                for b in np.array_split(X, max(1, len(X) // 8192 + 1))
            ]
        )
    return out.reshape(t.shape)


# Давление в центрах грубой сетки КО
TT, RR, ZZ = np.meshgrid(times, fv["rc"], fv["zc"], indexing="ij")
pp = p0 * a(ZZ) * predict_q(TT, RR, ZZ)  # PINN
pref = p0 * a(ZZ) * qfine  # эталон
weights = fv["V"][None, :, :]
rel = np.sqrt(
    np.sum(weights * (pp - pref) ** 2, axis=(1, 2))
    / np.sum(weights * pref**2, axis=(1, 2))
)
maxerr = np.max(np.abs(pp - pref), axis=(1, 2))

# Дебит PINN: закон Дарси на стенке скважины, интеграл по высоте квадратурой Гаусса — Лежандра
gz, gw = np.polynomial.legendre.leggauss(32)
zquad = h * (gz + 1) / 2
rates = []
for t in times:
    X = torch.tensor(
        np.c_[
            np.zeros_like(zquad), zquad / h, np.full_like(zquad, np.log1p(t / t0) / S)
        ],
        requires_grad=True,
    )
    q = model(X)
    qx = grad(q, X)[:, 0].detach().numpy()
    qv = q.detach().numpy().ravel()
    rates.append(
        2
        * np.pi
        * k
        * c
        * p0**2
        / (mu * L)
        * np.sum(gw * h / 2 * a(zquad) ** 2 * qv * qx)
    )
rates = np.array(rates)
rate_error = float(np.linalg.norm(rates - fine["rate"]) / np.linalg.norm(fine["rate"]))

# Баланс массы PINN: запасы по ячейкам тонкой сетки + интеграл дебита
Tf, Rf, Zf = np.meshgrid(times, fine["rc"], fine["zc"], indexing="ij")
mass_pinn = (predict_q(Tf, Rf, Zf) * fine["storage"]).sum(axis=(1, 2))
produced_pinn = np.r_[0, np.cumsum(0.5 * (rates[1:] + rates[:-1]) * np.diff(times))]
pinn_balance = float(
    np.max(np.abs(mass_pinn + produced_pinn - mass_pinn[0])) / mass_pinn[0]
)

checks = [
    ("макс. ошибка давления / депрессия", maxerr.max() / (delta * p0), 0.05),
    ("объёмно-взвеш. отн. ошибка давления", rel.max(), 0.01),
    ("отн. L2-ошибка дебита", rate_error, 0.10),
    ("невязка баланса массы / запасы", pinn_balance, 0.01),
]
print(
    f"макс. ошибка давления: {maxerr.max() / 1e3:.1f} кПа "
    f"(разница двух сеток КО: {refine_max:.1f} кПа)\n"
)
print(f"{'метрика':38} {'PINN':>9} {'цель':>7}")
for name, value, target in checks:
    print(
        f"{name:38} {value:9.2%} {'< ' + f'{target:.0%}':>7}  {'✓' if value < target else '✗'}"
    )
passed = all(value < target for _, value, target in checks)
print(
    "\nЦели",
    "ВЫПОЛНЕНЫ"
    if passed
    else "НЕ ВЫПОЛНЕНЫ: доработайте обучение, прежде чем верить прогнозу",
)

**Что мы видим.** Все четыре цели выполнены. Максимальная ошибка давления — 50 кПа, то есть 1% депрессии и 0.25% абсолютного давления. Это в 50 раз больше разницы двух сеток КО, так что мы измеряем ошибку PINN, а не эталона. Дебит зависит от производной на скважине, но и он совпал с точностью 0.6%. Баланс массы PINN никто не навязывал, и выполнен он с точностью 0.24% запасов.

Последнее число стоит прочитать внимательнее. За год отбирается 6.8% запасов, поэтому невязка баланса — это около **3.5% добытого газа**. Цель «1% запасов» оказалась мягкой: она выполнена, но для подсчёта добычи такой точности может не хватить. Цели надо формулировать в тех величинах, которые нужны инженеру.

И ещё: это одна инициализация и один бюджет обучения. Прежде чем опираться на цифры, повторите обучение с несколькими сидами (упражнение 1).

Посмотрим на решения рядом.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 8.5), constrained_layout=True)
j_mid = len(fv["zc"]) // 2
for days, col in zip(SHOW_DAYS, TIME_COLORS):
    it = np.argmin(abs(times - days * day))
    ax[0, 0].semilogx(fv["rc"], pref[it, :, j_mid] / 1e6, color=col, lw=4, alpha=0.45)
    ax[0, 0].semilogx(
        fv["rc"], pp[it, :, j_mid] / 1e6, "--", color=col, lw=1.8, label=f"{days} сут"
    )
ax[0, 0].plot([], [], color=INK_COLOR, lw=4, alpha=0.45, label="КО (эталон)")
ax[0, 0].plot([], [], "--", color=INK_COLOR, lw=1.8, label="PINN")
ax[0, 0].set(
    xlabel="r, м", ylabel="давление, МПа", title="Давление в середине пласта по высоте"
)
ax[0, 0].legend(fontsize=8, loc="lower right")

late = times >= 60.0
ax[0, 1].semilogx(
    times[late] / day, fine["rate"][late], label="КО (эталон)", **REFERENCE_STYLE
)
ax[0, 1].semilogx(times[late] / day, rates[late], color=PINN_COLOR, lw=2, label="PINN")
ax[0, 1].set(xlabel="время, сут", ylabel="дебит, кг/с", title="Дебит скважины")
ax[0, 1].legend()

ax[1, 0].semilogx(
    times[1:] / day, maxerr[1:] / 1e3, color=PINN_COLOR, lw=2, label="PINN − КО"
)
ax[1, 0].axhline(
    refine_max, color=MLP_COLOR, ls="--", lw=1.5, label="разница двух сеток КО (макс.)"
)
ax[1, 0].set(
    xlabel="время, сут", ylabel="кПа", title="Максимальная ошибка давления по пласту"
)
ax[1, 0].legend()

ax[1, 1].loglog(
    np.arange(1, len(history) + 1), history, color=PINN_COLOR, lw=1, alpha=0.8
)
ax[1, 1].axhline(
    test_rms**2, color=INK_COLOR, ls=":", lw=1.5, label="после L-BFGS, тестовые точки"
)
ax[1, 1].set(
    xlabel="шаг Adam", ylabel="лосс (нормированная невязка²)", title="Обучение"
)
ax[1, 1].legend()
plt.show()

**Что мы видим.** - Профили давления PINN и КО на глаз неразличимы во все четыре момента времени. Дебит совпадает от запуска скважины до конца года.
- Ошибка давления растёт со временем: в первые сутки 10–17 кПа, а максимум 50 кПа приходится на ~95-е сутки, вскоре после того, как волна депрессии дошла до границы. На всём интервале она намного выше разницы сеток КО (бирюзовая линия).
- Лосс за 5000 шагов Adam упал на четыре порядка, L-BFGS дал ещё порядок. Но по графику лосса нельзя сказать, где и когда PINN ошибается: это видно только при сравнении с эталоном.

### 5.1 Слабый сигнал: гравитация

В покое перепад давления по всей толщине пласта — 36 кПа, между центрами нижнего и верхнего слоёв ячеек — 31.5 кПа. Это величина того же порядка, что и ошибка PINN. Посмотрим, воспроизводит ли PINN именно этот перепад.

In [ ]:
t_grav = 180 * day
it = np.argmin(abs(times - t_grav))
zb, zt = fv["zc"][0], fv["zc"][-1]  # центры нижнего и верхнего слоёв ячеек
rplot = np.geomspace(fv["rc"][0], fv["rc"][-1], 200)
dp_pinn = p0 * (
    a(zb) * predict_q(t_grav, rplot, zb) - a(zt) * predict_q(t_grav, rplot, zt)
)
dp_fv = p0 * (a(zb) * qfine[it, :, 0] - a(zt) * qfine[it, :, -1])
dp_static = p0 * (a(zb) - a(zt))

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(fv["rc"], dp_fv / 1e3, label="КО (эталон)", **REFERENCE_STYLE)
ax.semilogx(rplot, dp_pinn / 1e3, color=PINN_COLOR, lw=2, label="PINN")
ax.axhline(dp_static / 1e3, color=INK_COLOR, ls=":", lw=1.5, label="в покое, t = 0")
ax.set(
    xlabel="r, м",
    ylabel="перепад давления, кПа",
    title=f"Перепад давления подошва − кровля, {t_grav / day:.0f} сут",
)
ax.legend()
plt.show()
print(
    f"перепад в покое {dp_static / 1e3:.2f} кПа; через {t_grav / day:.0f} сут у границы: "
    f"КО {dp_fv[-1] / 1e3:.2f} кПа, PINN {dp_pinn[-1] / 1e3:.2f} кПа"
)

**Что мы видим.** Через полгода перепад у скважины уменьшился с 31.5 до 23.7 кПа, а у границы — до 30.5 кПа: где давление ниже, там газ легче, и его столб весит меньше. PINN воспроизводит эту зависимость с ошибкой около 0.1 кПа, хотя весь сигнал меньше 1% депрессии.

Основную часть этой зависимости даёт уже замена переменных: если $q$ не зависит от высоты, перепад равен $q\cdot31.5$ кПа, и сети не нужно ничего учить. Это ещё один пример того, как постановка задачи делает работу за сеть: гидростатика встроена в $a(z)$, а сеть учит только малую поправку.

### 5.2 Где ошибается PINN?

Слева — ошибка давления в долях депрессии (максимум по высоте) на плоскости $(r,t)$, красная линия — радиус исследования. Справа — та же ошибка по радиусу в четыре момента времени и, на правой оси, накопленные доли точек коллокации и объёма пласта внутри радиуса $r$. Если ошибка растёт там, где точек уже мало, а газа много, то узкое место — выборка точек.

In [ ]:
err_rt = np.max(np.abs(pp - pref), axis=2) / (delta * p0)  # (время, r), максимум по z
fig, ax = plt.subplots(1, 2, figsize=(15, 4.6), constrained_layout=True)
im = ax[0].pcolormesh(
    fv["rc"],
    times[1:] / day,
    np.clip(err_rt[1:], 1e-6, None),
    norm=LogNorm(1e-5, max(1e-2, err_rt.max())),
    cmap="Blues",
    shading="nearest",
)
ax[0].plot(
    np.sqrt(4 * D * times[1:]),
    times[1:] / day,
    color=REFERENCE_COLOR,
    lw=2,
    label=r"$\sqrt{4Dt}$",
)
ax[0].set(
    xscale="log",
    yscale="log",
    xlim=(fv["rc"][0], fv["rc"][-1]),
    xlabel="r, м",
    ylabel="t, сут",
    title="Ошибка PINN / депрессия, максимум по z",
)
ax[0].legend(loc="lower right")
ax[0].grid(False)
fig.colorbar(im, ax=ax[0], label="|ошибка p| / депрессия")

for days, col in zip(SHOW_DAYS, TIME_COLORS):
    it = np.argmin(abs(times - days * day))
    ax[1].loglog(
        fv["rc"], np.clip(err_rt[it], 1e-6, None), color=col, lw=2, label=f"{days} сут"
    )
ax[1].set(xlabel="r, м", ylabel="ошибка / депрессия", title="Ошибка по радиусу")
ax[1].legend(loc="upper left", fontsize=8)
ax2 = ax[1].twinx()
ax2.semilogx(
    fv["rc"],
    np.log(fv["rc"] / rw) / L,
    "--",
    color=INK_COLOR,
    label="доля точек коллокации с радиусом < r",
)
ax2.semilogx(
    fv["rc"],
    (fv["rc"] ** 2 - rw**2) / (R**2 - rw**2),
    ":",
    color=INK_COLOR,
    label="доля объёма пласта с радиусом < r",
)
ax2.set_ylabel("накопленная доля")
ax2.grid(False)
ax2.legend(loc="lower right", fontsize=8)
plt.show()

it = np.unravel_index(np.argmax(err_rt), err_rt.shape)
print(
    f"макс. ошибка {err_rt.max():.2%} депрессии при r = {fv['rc'][it[1]]:.0f} м, "
    f"t = {times[it[0]] / day:.0f} сут"
)

**Что мы видим.** - Ниже красной линии ошибка практически нулевая: волна депрессии туда ещё не дошла, решение $q\approx1$, и форма $q$ с множителем $e^{-4\xi}$ задаёт его почти точно с самого начала.
- Самая большая ошибка, 1% депрессии, — у внешней границы ($r\approx930$ м) на 95-е сутки, вскоре после прихода волны. У скважины, где решение меняется быстрее всего, ошибка в несколько раз меньше.
- Справа видно, почему: при $r>100$ м доля точек коллокации растёт медленно и линейно по $\ln r$, а доля газа — от 1% до 100%. PINN ошибается там, где **газа много, а точек мало**. Узкое место — выборка точек и бюджет обучения, а не постановка задачи (упражнения 2 и 3).

## Выводы

- **PINN справилась без данных.** Обученная только на уравнении и условиях, она воспроизвела давление (ошибка не больше 1% депрессии), дебит (0.6%) и слабый эффект гравитации в задаче, у которой нет аналитического решения. Ни одного значения из расчёта КО в обучении не было.
- **Работу сделала постановка, а не размер сети.** Замена переменных убрала гидростатику, логарифмические координаты покрыли четыре порядка по радиусу и семь по времени, все условия встроены в форму решения, невязка нормирована на локальный масштаб. Это те же правила, что в ноутбуке 2: обезразмеривать, выводить масштабы из физики, встраивать условия.
- **Эталон тоже надо проверять.** Равновесие, баланс массы и измельчение сетки показали, что ошибка эталона около 1 кПа. Без этого 50 кПа ошибки PINN нечем было бы измерить.
- **Проверяйте по физическим величинам.** Ошибка PINN сосредоточена там, где мало точек коллокации и много газа, а невязка баланса массы — 3.5% добытого газа. Ни то, ни другое не видно по лоссу.
- **Честное сравнение.** КО решает эту задачу за доли секунды и точнее, PINN обучается минуту. Для прямой задачи с известными параметрами классический решатель лучше. Сильные стороны PINN в другом: решение непрерывно и дифференцируемо, сетка не нужна, а измерения и неизвестные параметры добавляются в тот же лосс (обратные задачи, см. упражнение 6 ноутбука 2).

## Упражнения

1. **Сиды.** Обучите PINN с `torch.manual_seed` = 1, 2, 3 (перед созданием `model`). Насколько меняется максимальная ошибка давления? Устойчивы ли выводы? Не выбирайте лучший сид по ошибке относительно эталона: в реальной задаче эталона нет.
2. **Выборка точек.** Добавьте к точкам Соболя часть точек, равномерных по $r^2$ (то есть по объёму газа) и по $t$. Уменьшается ли ошибка у внешней границы? Что происходит у скважины?
3. **Бюджет.** Увеличьте `ADAM_STEPS` до 20000 и `N_COLLOCATION` до 4096. Как меняются лосс, ошибка давления и ошибка дебита? Падают ли они согласованно?
4. **Эталон.** Посчитайте КО на сетке 256×32 и с `rtol=1e-9`. Насколько изменилась разница сеток? Хватает ли точности эталона, чтобы измерять ошибку PINN?
5. **Без гравитации.** Положите `g = 0` и перезапустите **все** ячейки. Решение должно перестать зависеть от высоты. Проверьте это и для КО, и для PINN.
6. **Проницаемость.** Уменьшите `k` в 10 раз. Успеет ли волна депрессии дойти до границы за год? Что нужно поменять в PINN, а что в КО?
7. **Сложнее.** Сделайте вязкость зависящей от давления, $\mu(p)=\mu_0(1+\gamma(p-p_0)/p_0)$. Какие места в выводе раздела 1.2, в схеме КО и в невязке PINN нужно изменить? Почему важно менять оба решателя согласованно?

**Источники:** [интегратор BDF в SciPy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.BDF.html), [автоматическое дифференцирование в PyTorch](https://docs.pytorch.org/docs/stable/autograd.html); метод конечных объёмов для фильтрации — K. Aziz, A. Settari, *Petroleum Reservoir Simulation* (1979); обзор приёмов обучения PINN — [S. Wang et al., *An Expert's Guide to Training Physics-informed Neural Networks*](https://arxiv.org/abs/2308.08468) (2023).